In [ ]:
import pandas as pd
import numpy as np
import joblib
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

import matplotlib.pyplot as plt
from sklearn.model_selection import KFold, cross_val_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
features = pd.read_csv("../data/customer_features.csv")
train = pd.read_csv("../data/customer_clv_train.csv")

# merge customer_features with training data
df = train.merge(features, on="cust_id", how="left")

In [ ]:
churn_lgb = joblib.load("../models/churn_lgb_model.pkl")
churn_xgb = joblib.load("../models/churn_xgb_model.pkl")
churn_cat = joblib.load("../models/churn_cat_model.pkl")

iso_lgb = joblib.load("../models/iso_lgb.pkl")
iso_xgb = joblib.load("../models/iso_xgb.pkl")
iso_cat = joblib.load("../models/iso_cat.pkl")

# definition churn classifier blend (equal weights)
def predict_churn_proba(X):
    p_lgb = iso_lgb.transform(churn_lgb.predict_proba(X)[:, 1])
    p_xgb = iso_xgb.transform(churn_xgb.predict_proba(X)[:, 1])
    p_cat = iso_cat.transform(churn_cat.predict_proba(X)[:, 1])
    return (p_lgb + p_xgb + p_cat) / 3

In [ ]:
feature_cols = joblib.load("../models/feature_columns.pkl")

In [ ]:
# fixed 3-ways split 60% df_train, 20% df_thresh, 20% df_val
df_trainval, df_val = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_thresh = train_test_split(df_trainval, test_size=0.25, random_state=42)
print(f"Train: {len(df_train)}  Thresh: {len(df_thresh)}  Val: {len(df_val)}")

# churn prediction
p_return_val = predict_churn_proba(df_val[feature_cols])

# separating true returners for training of revenue model 2-stage
df_return_train = df_train[df_train["revenue_2018_2019"] > 0].copy()
X_rev_train = df_return_train[feature_cols]

# target transformation sqrt
y_rev_train = np.sqrt(df_return_train["revenue_2018_2019"].values)

## Revenue predictor 2-stage model

In [ ]:
# lgb revenue predictor 2-stage; fixed hyperparameters
lgb_model = lgb.LGBMRegressor(
    objective="mae",
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=64,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_rev_train, y_rev_train)

In [ ]:
# xgb revenue predictor 2-stage; fixed hyperparameters
xgb_model = xgb.XGBRegressor(
    objective="reg:absoluteerror",
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_rev_train, y_rev_train)

In [ ]:
# catb revenue predictor 2-stage; fixed hyperparameters
cat_model = CatBoostRegressor(
    iterations=800,
    learning_rate=0.03,
    depth=6,
    loss_function="MAE",
    verbose=False,
    random_state=42
)

cat_model.fit(X_rev_train, y_rev_train)

In [ ]:
# optimization of threshold on df_thresh
p_return_thresh = predict_churn_proba(df_thresh[feature_cols])

# spearman_factor (trade-off Spearman vs. MAE) - fixed set to 0
SPEARMAN_FACTOR = 0  

best_overall_score = 1000
best_overall_mae = 1000
best_threshold = 0
rev_model = None
best_model_name = None

for name, model in [("LightGBM", lgb_model), ("XGBoost", xgb_model), 
                    ("CatBoost", cat_model)]:
    
    log_preds_thresh = model.predict(df_thresh[feature_cols])
    rev_preds_thresh = log_preds_thresh ** 2
    
    best_corr, best_t, best_mae = -1, 0, 1000
    best_score = 1000
    
    for t in np.arange(0.10, 0.91, 0.01):
        preds = np.where(p_return_thresh < t, 0, p_return_thresh * rev_preds_thresh)
        mae = mean_absolute_error(df_thresh["revenue_2018_2019"], preds)
        corr, _ = spearmanr(df_thresh["revenue_2018_2019"], preds)
        score = mae - SPEARMAN_FACTOR * corr
        if score < best_score:
            best_score = score
            best_corr, best_t, best_mae = corr, t, mae
    
    print(f"{name} - Thresh MAE: {best_mae:.2f}  Spearman: {best_corr:.4f}  Threshold: {best_t:.2f}")
    
    if best_score < best_overall_score:
        best_overall_score = best_score
        best_overall_mae = best_mae
        best_threshold = best_t
        rev_model = model
        best_model_name = name

print(f"\nBest model: {best_model_name}  Threshold: {best_threshold:.2f}")

# final validation MAE on df_val 
log_preds_val = rev_model.predict(df_val[feature_cols])
rev_preds_val = log_preds_val ** 2
final_val_preds = np.where(p_return_val < best_threshold, 0, p_return_val * rev_preds_val)
val_mae = mean_absolute_error(df_val["revenue_2018_2019"], final_val_preds)
val_corr, _ = spearmanr(df_val["revenue_2018_2019"], final_val_preds)
print(f"\nFinal MAE on df_val: {val_mae:.2f}  Spearman: {val_corr:.4f}")

In [ ]:
# Retrain best revenue model 2-stage model on full trainval (80% of the data) 
# (Threshold and model choice already are fixed on df_thresh)
df_return_trainval = df_trainval[df_trainval["revenue_2018_2019"] > 0].copy()
X_rev_full = df_return_trainval[feature_cols]
y_rev_full = np.sqrt(df_return_trainval["revenue_2018_2019"].values)

print(f"Returners voor retrain: {len(df_return_trainval)} "
      f"(was {len(df_return_train)} op 60%)")

# Retrain with exactly same parameters as the winning model
if best_model_name == "LightGBM":
    rev_model_final = lgb.LGBMRegressor(
        **rev_model.get_params(), random_state=42, verbose=-1
    )
elif best_model_name == "XGBoost":
    params = rev_model.get_params()
    params.pop("random_state", None)
    params.pop("verbosity", None)
    rev_model_final = xgb.XGBRegressor(
        **params, random_state=42, verbosity=0
    )
elif best_model_name == "CatBoost":
    params = rev_model.get_params()
    params.pop("random_seed", None)
    params.pop("random_state", None)
    params.pop("verbose", None)
    rev_model_final = CatBoostRegressor(
        **params, random_seed=42, verbose=False
    )

rev_model_final.fit(X_rev_full, y_rev_full)

# Verification on val-set - only for info
val_pred_final = np.where(
    p_return_val < best_threshold,
    0,
    p_return_val * np.maximum(rev_model_final.predict(df_val[feature_cols]) ** 2, 0)
    )

mae_final = np.mean(np.abs(val_pred_final - df_val["revenue_2018_2019"].values))
spearman_final = pd.Series(val_pred_final).corr(
    pd.Series(df_val["revenue_2018_2019"].values), method="spearman"
)
print(f"Val MAE after retrain:     {mae_final:.2f}  (was {val_mae:.2f})")
print(f"Val Spearman after retrain: {spearman_final:.4f}  (was {val_corr:.4f})")

In [ ]:
# save rev_model 2-stage + best threshold
joblib.dump(rev_model_final, "../models/rev_model_2stage.pkl")
joblib.dump(best_threshold, "../models/best_threshold.pkl")

## Pure regressor blend 
final_pred = β × pure_pred + (1-β) × two_stage_pred

In [ ]:
# Pure regressor trains on all customers of df_train (churners + returners)
# (only on 60%, no retraining on 80%)
X_pure_train = df_train[feature_cols]

# target transformation (sqrt)
y_pure_train = np.sqrt(df_train["revenue_2018_2019"].values)

cv_opt_pure = KFold(n_splits=3, shuffle=True, random_state=0)

# LGB pure regressor training + Optuna opt
def objective_pure_lgb(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 150),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    model = lgb.LGBMRegressor(
        objective="regression_l1", random_state=42,
        n_jobs=-1, verbose=-1, **params
    )
    scores = cross_val_score(
        model, X_pure_train, y_pure_train,
        cv=cv_opt_pure, scoring="neg_mean_absolute_error", n_jobs=1
    )
    return -scores.mean()

study_pure_lgb = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_pure_lgb.optimize(objective_pure_lgb, n_trials=30, show_progress_bar=True)
best_pure_lgb_params = study_pure_lgb.best_params
print(f"Pure LGB best params: {best_pure_lgb_params}")

pure_lgb = lgb.LGBMRegressor(
    objective="regression_l1", random_state=42,
    n_jobs=-1, verbose=-1, **best_pure_lgb_params
)
pure_lgb.fit(X_pure_train, y_pure_train)

In [ ]:
## Pure regressors XGBoost and Catboost (with Optuna)
# XGB pure regressor training + Optuna opt
def objective_pure_xgb(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma":             trial.suggest_float("gamma", 1e-8, 1.0, log=True),
    }
    model = xgb.XGBRegressor(
        objective="reg:absoluteerror", random_state=42,
        n_jobs=-1, verbosity=0, **params
    )
    scores = cross_val_score(
        model, X_pure_train, y_pure_train,
        cv=cv_opt_pure, scoring="neg_mean_absolute_error", n_jobs=1
    )
    return -scores.mean()

study_pure_xgb = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_pure_xgb.optimize(objective_pure_xgb, n_trials=30, show_progress_bar=True)
best_pure_xgb_params = study_pure_xgb.best_params
print(f"Pure XGB best params: {best_pure_xgb_params}")

pure_xgb = xgb.XGBRegressor(
    objective="reg:absoluteerror", random_state=42,
    n_jobs=-1, verbosity=0, **best_pure_xgb_params
)
pure_xgb.fit(X_pure_train, y_pure_train)

# CAT pure regressor training + Optuna opt
def objective_pure_cat(trial):
    params = {
        "iterations":        trial.suggest_int("iterations", 300, 2000),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "depth":             trial.suggest_int("depth", 3, 10),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "l2_leaf_reg":       trial.suggest_float("l2_leaf_reg", 1e-8, 10.0, log=True),
    }
    model = CatBoostRegressor(
        loss_function="MAE", random_state=42,
        verbose=False, **params
    )
    scores = cross_val_score(
        model, X_pure_train, y_pure_train,
        cv=cv_opt_pure, scoring="neg_mean_absolute_error", n_jobs=1
    )
    return -scores.mean()

study_pure_cat = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_pure_cat.optimize(objective_pure_cat, n_trials=20, show_progress_bar=True)
best_pure_cat_params = study_pure_cat.best_params
print(f"Pure CAT best params: {best_pure_cat_params}")

pure_cat = CatBoostRegressor(
    loss_function="MAE", random_state=42,
    verbose=False, **best_pure_cat_params
)
pure_cat.fit(X_pure_train, y_pure_train)

In [ ]:
# preparation for optimization of beta for blend 2-stage model with pure regressors
# calculation of 2-stage prediction on df_thresh (purely for optimization of beta)
log_preds_thresh_final = rev_model.predict(df_thresh[feature_cols])
rev_preds_thresh_final = log_preds_thresh_final ** 2
two_stage_thresh = np.where(
    p_return_thresh < best_threshold,
    0,
    p_return_thresh * rev_preds_thresh_final
)

In [ ]:
# pure regressors: optimization of beta on df_thresh per algorithm + validation on df_val
pure_preds_thresh_opt = np.maximum(pure_lgb.predict(df_thresh[feature_cols]) ** 2, 0)
pure_preds_val_opt    = np.maximum(pure_lgb.predict(df_val[feature_cols]) ** 2, 0)

pure_preds_val_xgb = np.maximum(pure_xgb.predict(df_val[feature_cols]) ** 2, 0)
pure_preds_thresh_xgb = np.maximum(pure_xgb.predict(df_thresh[feature_cols]) ** 2, 0)

pure_preds_val_cat = np.maximum(pure_cat.predict(df_val[feature_cols]) ** 2, 0)
pure_preds_thresh_cat = np.maximum(pure_cat.predict(df_thresh[feature_cols]) ** 2, 0)

beta_range = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print(f"{'Model':<12} {'β':<6} {'Val MAE':>10} {'Val Spearman':>14}")
print("-" * 44)

best_betas = {}

for name, pure_preds_thresh_m, pure_preds_val_m in [
    ("Pure LGB", pure_preds_thresh_opt, pure_preds_val_opt),
    ("Pure XGB", pure_preds_thresh_xgb, pure_preds_val_xgb),
    ("Pure CAT", pure_preds_thresh_cat, pure_preds_val_cat)
]:
    best_b = 0
    best_mae_t = 1000
    for beta in beta_range:
        blend_t = beta * pure_preds_thresh_m + (1 - beta) * two_stage_thresh
        mae_t = mean_absolute_error(df_thresh["revenue_2018_2019"], blend_t)
        if mae_t < best_mae_t:
            best_mae_t = mae_t
            best_b = beta

    best_betas[name] = best_b

    blend_v = best_b * pure_preds_val_m + (1 - best_b) * val_pred_final
    mae_v = mean_absolute_error(df_val["revenue_2018_2019"], blend_v)
    corr_v, _ = spearmanr(df_val["revenue_2018_2019"], blend_v)
    print(f"{name:<12} {best_b:<6.2f} {mae_v:>10.2f} {corr_v:>14.4f}")

print(f"\nBaseline β=0.00: MAE {mae_final:.2f}  Spearman {spearman_final:.4f}")

In [ ]:
# saving pure regressors and best betas
joblib.dump(pure_xgb, "../models/pure_xgb.pkl")
joblib.dump(pure_cat, "../models/pure_cat.pkl")
joblib.dump(pure_lgb, "../models/pure_lgb.pkl")
joblib.dump(best_betas["Pure LGB"], "../models/best_beta_lgb.pkl")
joblib.dump(best_betas["Pure XGB"], "../models/best_beta_xgb.pkl")
joblib.dump(best_betas["Pure CAT"], "../models/best_beta_cat.pkl")

# printing of best betas for each pure regressor
print(f"  LGB:      {best_betas['Pure LGB']:.2f}")
print(f"  XGB:      {best_betas['Pure XGB']:.2f}")
print(f"  CAT:      {best_betas['Pure CAT']:.2f}")

In [ ]:
# β optimization plot
beta_range = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

results_plot = {
    "Pure LGB":      (pure_preds_thresh_opt,      pure_preds_val_opt),
    "Pure XGB":      (pure_preds_thresh_xgb,      pure_preds_val_xgb),
    "Pure CAT":      (pure_preds_thresh_cat,       pure_preds_val_cat)
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (thresh_preds, val_preds) in results_plot.items():
    val_maes, val_spearmans = [], []
    for beta in beta_range:
        blend_v = beta * val_preds + (1 - beta) * val_pred_final
        val_maes.append(mean_absolute_error(df_val["revenue_2018_2019"], blend_v))
        val_spearmans.append(spearmanr(df_val["revenue_2018_2019"], blend_v)[0])
    axes[0].plot(beta_range, val_maes, marker="o", label=name)
    axes[1].plot(beta_range, val_spearmans, marker="o", label=name)

axes[0].set_xlabel("β")
axes[0].set_ylabel("Val MAE")
axes[0].set_title("Val MAE vs β — 3 pure regressors")
axes[0].legend()
axes[0].grid(True)

axes[1].set_xlabel("β")
axes[1].set_ylabel("Val Spearman")
axes[1].set_title("Val Spearman vs β — 3 pure regressors")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Optuna plot pure regressors
for study, name in [(study_pure_lgb, "LightGBM"), 
                    (study_pure_xgb, "XGBoost"), 
                    (study_pure_cat, "CatBoost")]:
    trials = [t.value for t in study.trials]
    best_vals = [min(trials[:i+1]) for i in range(len(trials))]
    plt.plot(best_vals, label=name)

plt.xlabel("Trial")
plt.ylabel("Best CV MAE")
plt.title("Pure Regressors — Best MAE per trial")
plt.legend()
plt.show()

In [ ]:
# consistency check: churn proobabilities are stable? 
print(f"\n--- Consistency check ---")
print(f"Average churn probability (val):       {p_return_val.mean():.4f}")
print(f"Fraction predicted as returner:    {(p_return_val > best_threshold).mean():.4f}")
print(f"Actual fraction of returners:      {(df_val['revenue_2018_2019'] > 0).mean():.4f}")